# Passing, rushing, and scoring margin

Predict scoring margin per game from offensive and defensive passing and rushing efficiency.
Compare OLS without an intercept and XGBoost across three training windows
(2015–2024, 2020–2024, and 2022–2024), with the full 2025 regular season as the common test set.


## 1. Data preparation

Each observation is one team-season. Use the saved team-game data when available;
otherwise load it from nflverse. Keep regular-season games from 2015 through 2025.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nflreadpy as nfl
import numpy as np
import pandas as pd
import polars as pl
import statsmodels.formula.api as smf
from sklearn.metrics import r2_score

FIGURE_DIR = Path("figures")
FIGURE_DIR.mkdir(exist_ok=True)


In [ ]:
team_stats_path = Path("filtered_team_game_stats.csv")
team_stats = (
    pl.read_csv(team_stats_path, infer_schema_length=10000)
    if team_stats_path.exists()
    else nfl.load_team_stats(seasons=list(range(2015, 2026)))
)

filtered_team_stats = (
    team_stats
    .filter(
        (pl.col("season_type") == "REG")
        & pl.col("season").is_between(2015, 2025)
    )
    .select(
        "season", "season_type", "game_id", "team", "opponent_team",
        "passing_yards", "attempts", "sacks_suffered", "sack_yards_lost",
        "rushing_yards", "carries",
    )
)


### Passing and rushing efficiency

Calculate season-level rates from total yards and attempts. Join each team's opponent
from the same game to calculate the defensive rates.


In [ ]:
games = filtered_team_stats.filter(pl.col("season_type") == "REG")

# Get each opponent's offensive stats from the same game.
opponents = games.select(
    "game_id",
    pl.col("team").alias("opponent_team"),
    pl.col("passing_yards").alias("opp_passing_yards"),
    pl.col("attempts").alias("opp_attempts"),
    pl.col("sacks_suffered").alias("opp_sacks"),
    pl.col("sack_yards_lost").alias("opp_sack_yards_lost"),
    pl.col("rushing_yards").alias("opp_rushing_yards"),
    pl.col("carries").alias("opp_carries"),
)

games = games.join(
    opponents,
    on=["game_id", "opponent_team"],
    how="left",
    validate="1:1",
)

assert games["opp_attempts"].null_count() == 0, "Missing opponent game rows"

season_stats = (
    games
    .group_by(["season", "team"])
    .agg(
        games_played=pl.col("game_id").n_unique(),

        pass_nya=(
            (pl.col("passing_yards").sum() + pl.col("sack_yards_lost").sum())
            / (pl.col("attempts").sum() + pl.col("sacks_suffered").sum())
        ),

        def_pass_nya=(
            (pl.col("opp_passing_yards").sum()
             + pl.col("opp_sack_yards_lost").sum())
            / (pl.col("opp_attempts").sum() + pl.col("opp_sacks").sum())
        ),

        rush_ypa=(
            pl.col("rushing_yards").sum() / pl.col("carries").sum()
        ),

        def_rush_ypa=(
            pl.col("opp_rushing_yards").sum() / pl.col("opp_carries").sum()
        ),
    )
    .sort(["season", "team"])
)


### Scoring margin

Join game scores and average each team's points scored minus points allowed over the season.


In [ ]:
scores = (
    nfl.load_schedules(seasons=list(range(2015, 2026)))
    .select("game_id", "home_team", "home_score", "away_score")
    .drop_nulls(["home_score", "away_score"])
)


In [ ]:
team_margins = (
    filtered_team_stats
    .filter(pl.col("season_type") == "REG")
    .join(scores, on="game_id", how="inner", validate="m:1")
    .with_columns(
        pl.when(pl.col("team") == pl.col("home_team"))
        .then(pl.col("home_score") - pl.col("away_score"))
        .otherwise(pl.col("away_score") - pl.col("home_score"))
        .alias("margin")
    )
    .group_by(["season", "team"])
    .agg(scoring_margin=pl.col("margin").mean())
)


In [ ]:
df = pd.DataFrame(
    season_stats
    .join(team_margins, on=["season", "team"], how="inner")
    .to_dict(as_series=False)
)

df.head()


## 2. Training and test sets

Use the same four predictors for both models. Each training window ends in 2024;
all models are evaluated on the same 32 teams in 2025.


In [ ]:
features = ["pass_nya", "def_pass_nya", "rush_ypa", "def_rush_ypa"]

train_2015_2024 = df[df["season"].between(2015, 2024)].copy()
train_2020_2024 = df[df["season"].between(2020, 2024)].copy()
train_2022_2024 = df[df["season"].between(2022, 2024)].copy()
test = df[df["season"] == 2025].copy()

datasets = {
    "2015-2024": train_2015_2024,
    "2020-2024": train_2020_2024,
    "2022-2024": train_2022_2024,
}
expected_counts = {"2015-2024": 320, "2020-2024": 160, "2022-2024": 96}
assert len(test) == 32, "Expected all 32 teams in the 2025 test set"


for name, train in datasets.items():
    assert len(train) == expected_counts[name], f"Incomplete training data for {name}"


## 3. Correlations, 2015–2024

Pearson correlations among the four efficiency measures and scoring margin, using only
the 320 team-seasons in the 2015–2024 training window.


In [ ]:
from matplotlib.colors import LinearSegmentedColormap

correlation_matrix = train_2015_2024[features + ["scoring_margin"]].corr(method="pearson")
purple_cmap = LinearSegmentedColormap.from_list(
    "signed_purple", ["#9970ab", "#ffffff", "#542788"]
)

with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10, "svg.fonttype": "none"}):
    correlation_fig, correlation_ax = plt.subplots(figsize=(7.4, 6.2), facecolor="white")
    heatmap = correlation_ax.imshow(
        correlation_matrix.to_numpy(), cmap=purple_cmap, vmin=-1, vmax=1
    )
    correlation_ax.set_xticks(
        np.arange(len(correlation_matrix.columns)),
        correlation_matrix.columns,
        rotation=38,
        ha="right",
        rotation_mode="anchor",
    )
    correlation_ax.set_yticks(np.arange(len(correlation_matrix.index)), correlation_matrix.index)
    correlation_ax.tick_params(axis="both", which="both", length=0, pad=8)
    correlation_ax.set_title("Pearson correlations, 2015–2024", loc="left", fontsize=11, pad=14)
    for spine in correlation_ax.spines.values():
        spine.set_visible(False)

    # Keep cell boundaries subtle and display every signed correlation.
    correlation_ax.set_xticks(np.arange(-0.5, len(correlation_matrix.columns), 1), minor=True)
    correlation_ax.set_yticks(np.arange(-0.5, len(correlation_matrix.index), 1), minor=True)
    correlation_ax.grid(which="minor", color="white", linewidth=1)
    for row in range(len(correlation_matrix.index)):
        for column in range(len(correlation_matrix.columns)):
            value = correlation_matrix.iloc[row, column]
            red, green, blue, _ = heatmap.cmap(heatmap.norm(value))
            luminance = 0.299 * red + 0.587 * green + 0.114 * blue
            correlation_ax.text(
                column, row, f"{value:.3f}", ha="center", va="center", fontsize=10,
                color="white" if luminance < 0.52 else "#262626",
            )

    colorbar = correlation_fig.colorbar(heatmap, ax=correlation_ax, fraction=0.045, pad=0.045)
    colorbar.set_ticks([-1, -0.5, 0, 0.5, 1])
    colorbar.set_label("Pearson r", fontsize=10)
    colorbar.outline.set_visible(False)
    colorbar.ax.tick_params(length=0, labelsize=9)
    correlation_fig.tight_layout(pad=0.8)
    correlation_fig.savefig(FIGURE_DIR / "correlation_2015_2024.png", dpi=300, bbox_inches="tight", facecolor="white")
    correlation_fig.savefig(FIGURE_DIR / "correlation_2015_2024.svg", bbox_inches="tight", facecolor="white")
    plt.show()


## 4. OLS regression, 2015–2024

Keep the predictors in their original units and fit OLS without an intercept.
Coefficients describe the change in predicted scoring margin per game for a one-unit
change in a predictor, holding the others constant. Fit the shorter training windows
as well for the final comparison.


In [ ]:
formula = "scoring_margin ~ 0 + pass_nya + def_pass_nya + rush_ypa + def_rush_ypa"
models = {
    name: smf.ols(formula, data=train, missing="raise").fit()
    for name, train in datasets.items()
}

# Keep coefficients for all three windows available for inspection.
coefficients = pd.DataFrame({name: fitted.params for name, fitted in models.items()}).T
coefficients.index.name = "training_window"


In [ ]:
print("OLS regression summary: 2015–2024")
print(models["2015-2024"].summary())


In [ ]:
regression_2015_2024 = models["2015-2024"]
confidence_intervals = regression_2015_2024.conf_int()
regression_rows = []
for feature in regression_2015_2024.params.index:
    p_value = regression_2015_2024.pvalues[feature]
    regression_rows.append([
        feature,
        f"{regression_2015_2024.params[feature]:.4f}",
        f"{regression_2015_2024.bse[feature]:.3f}",
        f"{regression_2015_2024.tvalues[feature]:.3f}",
        "<0.001" if p_value < 0.001 else f"{p_value:.3f}",
        f"[{confidence_intervals.loc[feature, 0]:.3f}, {confidence_intervals.loc[feature, 1]:.3f}]",
    ])

with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10, "svg.fonttype": "none"}):
    regression_fig, regression_ax = plt.subplots(
        figsize=(8.8, 1.25 + 0.30 * (len(regression_rows) + 1)), facecolor="white"
    )
    regression_fig.subplots_adjust(left=0.025, right=0.975, top=0.83, bottom=0.21)
    regression_ax.set_axis_off()
    regression_table = regression_ax.table(
        cellText=regression_rows,
        colLabels=["Predictor", "Coef.", "Std. error", "t", "p", "95% CI"],
        colWidths=[0.25, 0.13, 0.14, 0.12, 0.12, 0.24],
        cellLoc="right",
        colLoc="right",
        bbox=[0, 0, 1, 1],
    )
    regression_table.auto_set_font_size(False)
    regression_table.set_fontsize(10)
    for (row, column), cell in regression_table.get_celld().items():
        cell.set_facecolor("white")
        cell.set_edgecolor("#777777")
        cell.set_linewidth(0.6)
        cell.visible_edges = ""
        cell.PAD = 0.10
        cell.get_text().set_color("#222222")
        if column == 0:
            cell.get_text().set_ha("left")
        if row == 0:
            cell.visible_edges = "BT"
            cell.get_text().set_weight("semibold")
        elif row == len(regression_rows):
            cell.visible_edges = "B"

    regression_fig.text(0.025, 0.94, "OLS regression, 2015–2024", fontsize=11, va="top", color="#222222")
    regression_fig.text(
        0.025, 0.07,
        f"N = {int(regression_2015_2024.nobs)}    ·    R² (uncentered) = {regression_2015_2024.rsquared:.3f}",
        fontsize=9, color="#555555",
    )
    regression_fig.savefig(FIGURE_DIR / "ols_2015_2024.png", dpi=300, bbox_inches="tight", facecolor="white")
    regression_fig.savefig(FIGURE_DIR / "ols_2015_2024.svg", bbox_inches="tight", facecolor="white")
    plt.show()


## 5. XGBoost

Fit the same four predictors for each training window, using 100 trees, a maximum depth
of 2, a learning rate of 0.05, and a fixed random seed of 42.


In [ ]:
from xgboost import XGBRegressor

xgb_models = {}

for name, train in datasets.items():
    xgb_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=100,
        max_depth=2,
        learning_rate=0.05,
        random_state=42,
    )
    xgb_model.fit(train[features], train["scoring_margin"])
    xgb_models[name] = xgb_model


### XGBoost results

Report each training window separately, using the same 2025 test set. Higher R² is
better; lower MAE and RMSE are better. Both error measures are in scoring-margin
points per game. R² is a regression score, not a percentage of predictions that are correct.


In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

xgb_rows = []
for name, fitted in xgb_models.items():
    train = datasets[name]
    train_predictions = fitted.predict(train[features])
    test_predictions = fitted.predict(test[features])
    xgb_rows.append({
        "training_window": name,
        "train_r2": r2_score(train["scoring_margin"], train_predictions),
        "test_r2": r2_score(test["scoring_margin"], test_predictions),
        "test_mae": mean_absolute_error(test["scoring_margin"], test_predictions),
        "test_rmse": root_mean_squared_error(test["scoring_margin"], test_predictions),
    })

xgb_results = pd.DataFrame(xgb_rows)
xgb_results.round(3)


### Feature importance, 2015–2024

Tree-based XGBoost has no OLS-style coefficient, standard-error, or p-value table.
Instead, inspect each feature's average split gain in the fitted model, normalized
to sum to 100%. These are training-based importance scores; they do not indicate
effect direction, statistical significance, or causal effects.


In [ ]:
gain_scores = xgb_models["2015-2024"].get_booster().get_score(importance_type="gain")
gain_importance = pd.Series(gain_scores, dtype=float).reindex(features, fill_value=0)

xgb_importance = (
    (100 * gain_importance / gain_importance.sum())
    .rename("normalized_gain_pct")
    .sort_values(ascending=False)
    .rename_axis("feature")
    .to_frame()
)
xgb_importance.round(2)


## 6. Final R² comparison

Calculate training and 2025 test R² with `r2_score` for both models. This uses the same
centered R² definition for every row. The no-intercept OLS summary above reports
uncentered R² instead.


In [ ]:
# One performance metric, calculated the same way for every model and dataset.
comparison_rows = []

for model_name, fitted_models in {"OLS": models, "XGBoost": xgb_models}.items():
    for name, fitted in fitted_models.items():
        train = datasets[name]
        comparison_rows.append({
            "model": model_name,
            "training_window": name,
            "train_team_seasons": len(train),
            "train_r2": r2_score(
                train["scoring_margin"], fitted.predict(train[features])
            ),
            "test_r2": r2_score(
                test["scoring_margin"], fitted.predict(test[features])
            ),
        })

comparison = pd.DataFrame(comparison_rows)
comparison.round(3)
